<a href="https://colab.research.google.com/github/Park-gpow/bigdataAn/blob/main/data(03-31_01).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import argparse
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TARGET_CANDIDATES = [
    'discharge_capacity (mAh/g)',
    'discharge_capacity',
]

ID_COLS = {'material_id', 'DOI', 'journal_name', 'chemical_formula'}
LIKELY_CATEGORICAL = {
    'Class', 'material_structure', 'synthesis_method', 'Li_source', 'Co_source',
    'Mn_source', 'Ni_source', 'electrolyte', 'counter_electrode', 'separator',
    'space_group_symbol'
}


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {}
    for col in df.columns:
        clean = col.strip()
        if clean in TARGET_CANDIDATES:
            rename_map[col] = 'discharge_capacity'
        elif clean == 'voltage_range(V)':
            rename_map[col] = 'voltage_range'
        else:
            rename_map[col] = clean
    return df.rename(columns=rename_map)


def split_voltage_range(df: pd.DataFrame) -> pd.DataFrame:
    if 'voltage_range' not in df.columns:
        return df
    vr = df['voltage_range'].astype(str).str.replace('~', ',', regex=False)
    parts = vr.str.extract(r'\s*([+-]?\d*\.?\d+)\s*[,/ ]+\s*([+-]?\d*\.?\d+)')
    if parts.notna().any().any():
        df['voltage_range_min'] = pd.to_numeric(parts[0], errors='coerce')
        df['voltage_range_max'] = pd.to_numeric(parts[1], errors='coerce')
        df['voltage_window'] = df['voltage_range_max'] - df['voltage_range_min']
    return df


def infer_variable_type(series: pd.Series) -> str:
    name = series.name
    if name in ID_COLS:
        return 'identifier'
    if name in {'remain_capacity', 'state_of_charge', 'Strain'}:
        return 'derived_or_control'
    if name in LIKELY_CATEGORICAL:
        return 'categorical'
    if pd.api.types.is_numeric_dtype(series):
        return 'numeric'
    return 'categorical'


def summarize_variables(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        s = df[col]
        vtype = infer_variable_type(s)
        row = {
            'variable': col,
            'type': vtype,
            'n': len(s),
            'missing_n': int(s.isna().sum()),
            'missing_pct': round(float(s.isna().mean() * 100), 4),
            'n_unique': int(s.nunique(dropna=True)),
        }
        if pd.api.types.is_numeric_dtype(s):
            row.update({
                'min': pd.to_numeric(s, errors='coerce').min(),
                'max': pd.to_numeric(s, errors='coerce').max(),
                'mean': pd.to_numeric(s, errors='coerce').mean(),
                'std': pd.to_numeric(s, errors='coerce').std(),
            })
        else:
            top = s.astype(str).value_counts(dropna=False).head(5)
            row['top_levels'] = '; '.join([f'{idx}:{val}' for idx, val in top.items()])
        rows.append(row)
    summary = pd.DataFrame(rows)
    summary['is_constant'] = summary['n_unique'].fillna(0).eq(1)
    summary['recommended_action'] = np.select(
        [
            summary['variable'].isin(['remain_capacity']),
            summary['variable'].isin(list(ID_COLS)),
            summary['is_constant'],
            summary['variable'].isin(['state_of_charge', 'Strain']),
        ],
        [
            'exclude_target_leakage',
            'exclude_identifier',
            'exclude_constant',
            'control_or_auxiliary_review',
        ],
        default='review'
    )
    return summary.sort_values(['type', 'variable']).reset_index(drop=True)


def category_levels(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        if infer_variable_type(df[col]) != 'categorical':
            continue
        vc = df[col].astype(str).value_counts(dropna=False)
        for level, count in vc.items():
            rows.append({
                'variable': col,
                'level': level,
                'count': int(count),
                'pct': float(count / len(df) * 100),
            })
    return pd.DataFrame(rows)


def numeric_summary(df: pd.DataFrame) -> pd.DataFrame:
    num_cols = [c for c in df.columns if infer_variable_type(df[c]) == 'numeric']
    if not num_cols:
        return pd.DataFrame()
    desc = df[num_cols].describe(percentiles=[0.25, 0.5, 0.75]).T
    desc.index.name = 'variable'
    return desc.reset_index()


def save_basic_plots(df: pd.DataFrame, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    if 'discharge_capacity' in df.columns:
        plt.figure(figsize=(8, 5))
        df['discharge_capacity'].dropna().hist(bins=30)
        plt.title('discharge_capacity histogram')
        plt.xlabel('discharge_capacity')
        plt.ylabel('count')
        plt.tight_layout()
        plt.savefig(out_dir / 'hist_discharge_capacity.png', dpi=150)
        plt.close()

    numeric_cols = [c for c in df.columns if infer_variable_type(df[c]) == 'numeric']
    numeric_cols = [c for c in numeric_cols if c != 'discharge_capacity'][:6]
    for col in numeric_cols:
        plt.figure(figsize=(8, 5))
        df[col].dropna().hist(bins=30)
        plt.title(f'{col} histogram')
        plt.xlabel(col)
        plt.ylabel('count')
        plt.tight_layout()
        plt.savefig(out_dir / f'hist_{col}.png', dpi=150)
        plt.close()


def compare_train_val(train: pd.DataFrame, val: pd.DataFrame) -> pd.DataFrame:
    common = [c for c in train.columns if c in val.columns]
    rows = []
    for col in common:
        if infer_variable_type(train[col]) == 'numeric' and infer_variable_type(val[col]) == 'numeric':
            rows.append({
                'variable': col,
                'train_mean': pd.to_numeric(train[col], errors='coerce').mean(),
                'val_mean': pd.to_numeric(val[col], errors='coerce').mean(),
                'train_std': pd.to_numeric(train[col], errors='coerce').std(),
                'val_std': pd.to_numeric(val[col], errors='coerce').std(),
            })
    return pd.DataFrame(rows)


def main():
    parser = argparse.ArgumentParser(description='LFP 5주차 준비용 변수 점검 스크립트')
    parser.add_argument('--train', required=True, help='LFP_train_dataset.csv 경로')
    parser.add_argument('--val', required=True, help='LFP_val_dataset.csv 경로')
    parser.add_argument('--outdir', default='week5_output', help='결과 저장 폴더')
    args = parser.parse_args()

    out_dir = Path(args.outdir)
    out_dir.mkdir(parents=True, exist_ok=True)

    train = pd.read_csv(args.train)
    val = pd.read_csv(args.val)

    train = split_voltage_range(standardize_columns(train))
    val = split_voltage_range(standardize_columns(val))

    train_summary = summarize_variables(train)
    val_summary = summarize_variables(val)
    train_numeric = numeric_summary(train)
    val_numeric = numeric_summary(val)
    train_levels = category_levels(train)
    val_levels = category_levels(val)
    compare_df = compare_train_val(train, val)

    with pd.ExcelWriter(out_dir / 'week5_variable_audit.xlsx', engine='openpyxl') as writer:
        train_summary.to_excel(writer, sheet_name='train_variable_summary', index=False)
        val_summary.to_excel(writer, sheet_name='val_variable_summary', index=False)
        train_numeric.to_excel(writer, sheet_name='train_numeric_summary', index=False)
        val_numeric.to_excel(writer, sheet_name='val_numeric_summary', index=False)
        train_levels.to_excel(writer, sheet_name='train_category_levels', index=False)
        val_levels.to_excel(writer, sheet_name='val_category_levels', index=False)
        compare_df.to_excel(writer, sheet_name='train_val_compare', index=False)

    save_basic_plots(train, out_dir / 'plots_train')
    save_basic_plots(val, out_dir / 'plots_val')

    review_notes = {
        'recommended_target': 'discharge_capacity',
        'exclude_first': ['material_id', 'DOI', 'journal_name', 'chemical_formula', 'remain_capacity'],
        'review_as_control': ['state_of_charge', 'Strain'],
        'note': 'LFP 단독 분석에서는 조성비 관련 변수 다수가 상수일 수 있으므로 summary 시트에서 n_unique=1 여부를 먼저 확인하세요.'
    }
    (out_dir / 'review_notes.json').write_text(json.dumps(review_notes, ensure_ascii=False, indent=2), encoding='utf-8')

    print(f'완료: {out_dir.resolve()}')
    print('생성 파일: week5_variable_audit.xlsx, review_notes.json, plots_train/, plots_val/')


if __name__ == '__main__':
    main()


In [3]:
!git clone https://github.com/Park-gpow/bigdata-an.git
%cd bigdata-an
!git add .
!git commit -m "update colab notebook"
!git push

fatal: destination path 'bigdata-an' already exists and is not an empty directory.
/content/bigdata-an
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@d0b264e90d32.(none)')
error: src refspec refs/heads/main does not match any
error: failed to push some refs to 'https://github.com/Park-gpow/bigdata-an.git'
